<a id="inicio"></a>
<br><br><br>
<div align="left">
<h1><font size=6>Pandas</font></h1>
    <h2><font color="#5E5E5E" size=5>8. Operaciones sobre conjuntos de elementos: agregación </font></h2>
<br><br><br><br>
</div>


<div align="right">
<font color="#B51700" size=4><b>Análisis y visualización de datos</b></font><br><br>
<font  size=3>Máster Universitario en Ingeniería Informática</font><br>
<font  size=3>Máster Universitario en Big Data y Computación en la Nube</font><br>
<font size=3>Máster Universitario en Inteligencia Artificial</font><br><br>
<font color="#5E5E5E" size=3><b>Luis de la Ossa  &nbsp;&nbsp;</b></font><span style="background-color:#B51700; color:white; padding:2px 6px; font-family:Arial, sans-serif;font-weight:bold;font-size:18px;">UCLM</span>
</div>

---

Además de las funciones de *transformación*, que operan a nivel de elemento, *pandas* implementa funciones de ***agregación***, en las que las operaciones se definen sobre conjuntos de elementos. Cuando estas operaciones se aplican a un `DataFrame`, generalmente se aplican a las columnas o filas por separado. 

In [ ]:
import pandas as pd
import numpy as np
pd.__version__

Para ilustrar los ejemplos de esta parte del tutorial, se utilizará el conjunto de datos `fifa19_prep.csv`, que incorpora las 200 primeras filas del conjunto original, con las transformaciones de datos que se hicieron en la primera parte del tutorial, y algunos valores perdidos.

In [ ]:
df_fifa = pd.read_csv('./data/fifa19_prep.csv', index_col=0)
df_fifa.head()

---
## <font color="#5E5E5E"> 1. Funciones resumen y estadísticas en `Series`</font>


Los objetos de tipo `Series` implementan una gran cantidad de métodos de agregación. En general, estos devuelven información sobre los datos en forma de resumen o medidas estadísticas de descripción. Por otra parte, la información devuelta puede ser un valor escalar, o una serie con distintos valores. El listado completo puede consultarse en la sección correspondiente de la API [**(+ inf )**](https://pandas.pydata.org/pandas-docs/stable/reference/series.html#computations-descriptive-stats).

<div class="alert alert-block alert-warning">

<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
__Importante__: La mayoría de funciones ignoran los valores `NaN`.
</div>

La información resumen relativa a la serie consiste en datos sobre el conjunto de los elementos. En muchos casos, consiste en un valor.

In [ ]:
df_fifa['Name'].count()                  # Número de elementos (no nulos)
#df_fifa['Age'].max()                    # Máximo valor
#df_fifa['Value (M)'].idxmin()           # Índice correspondiente al mínimo valor
#df_fifa['Value (M)'].sum()              # Suma de los valores
#df_fifa['Club'].nunique()               # Número de valores distintos
#df_fifa['Club'].any()                   # Si existe algún valor distinto de False
#df_fifa['Club'].all()                   # Si todos los valores son distintos de False

Con respecto a `Series.any()` y `Series.all()`, ambos métodos se utilizan con frecuencia para detectar valores perdidos.

In [ ]:
df_fifa['Club'].isna().any()                    # Si hay algún valor pedido
#df_fifa['Club'].notna().all()                  # Si todos los valores son distintos al valor perdido

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>  Para este ejemplo se han utilizado las funciones `Series.isna()` \ `Series.notna()` que devuelven otra serie con el mismo índice, y valores `True` o `False` dependiendo de si el elemento original es `NaN` o no. Se verá después. 
</div>

Otras veces, el resumen consiste, a su vez, en otra `Series`.

In [ ]:
df_fifa['Value (M)'].nsmallest(5)                    # 5 menores valores
#df_fifa['Club'].unique()                            # Valores únicos
#df_fifa['Value (M)'].rank(ascending=False)          # Posición de cada elemento en el orden
#df_fifa['Club'].value_counts()                      # Número de veces que aparece cada valor

Cuando el resultado es una `Series`, se pueden volver a invocar estos métodos. Por ejemplo, en la siguiente celda se muestran los 5 valores que más aparecen en la serie.

In [ ]:
#df_fifa['Club'].value_counts()
df_fifa['Club'].value_counts().nlargest(5)

Por defecto, el método `Series.value_counts()` ignora los valores `NaN`, aunque puede indicarse que no lo haga fijando el parámetro `dropna` a `False`.

In [ ]:
df_fifa['Club'].value_counts()
#df_fifa['Club'].value_counts(dropna=False)

Las medidas resumen que computan algunos métodos consisten en estadísticos descriptivos. Por defecto omiten los valores ausentes(`skipna=True`).

In [ ]:
df_fifa['Age'].mean()                  # Media
#df_fifa['Value (M)'].std()            # Desviación estándar
#df_fifa['Age'].median()               # Mediana

---
## <font color="#5E5E5E"> 2. Funciones resumen en `DataFrames`</font>


Las funciones (métodos) resumen para `DataFrames`, en algunos casos, son similares a las que implementan las `Series`, y se aplican a cada fila o columna en función del parámetro `axis`, que  puede tomar los valores `index/0` o `columns/1`. El resultado es una `Series`. Además, muchos métodos aceptan (u obligan a usar)  el parámetro `numeric_only` que, cuando es `True`, indica que solo se opere sobre los datos numéricos.


<div class="alert alert-block alert-warning">

<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
`axis` indica el índice usado en la `Series` sobre la que se hace el cálculo. En este caso, corresponde al **no se mantiene**. 
</div>

In [ ]:
df_fifa.mean(numeric_only=True, axis=0)     # Media de cada columna. 
#df_fifa.std(numeric_only=True).            # Desviación de cada columna. 

Algunos métodos resumen que se implementan en `Series` y devuelven, a su vez, `Series`, se aplican a cada una de las columnas de forma independiente.

In [ ]:
df_fifa[['Overall', 'Value (M)']].rank(ascending=False)

La implementación de otros métodos en los `DataFrame` tiene un comportamiento específico. Por ejemplo, `DataFrame.value_counts()` devuelve el número de veces que aparece cada posible combinación de valores en las columnas seleccionadas. Se usa generalmente sobre conjuntos reducidos de columnas, y produce una `Series` que puede tener un índice de varios niveles (se verá después).

In [ ]:
df_fifa[['Club', 'Position']].value_counts()

Otro método, `DataFrame.nlargest()`, devuelve las filas que tienen mayor valor en la columna especificada. Si se especifican varias columnas, ordena por la primera, y usa la siguiente para desempatar, y así sucesivamente.

In [ ]:
df_fifa.nlargest(5,['Overall', 'Value (M)'])

---
## <font color="#5E5E5E"> 3. Agregación en *Series* y *DataFrames*</font>



### <font color="#5E5E5E">`Series.agg()`</font>

El método `Series.agg()` recibe un argumento que puede ser una función de agregación, o una lista de funciones de agregación, y devuelve un escalar, u otra `Series` con el resultado de cada una de ellas.

In [ ]:
df_fifa['Value (M)'].agg('mean')
#df_fifa['Value (M)'].agg([pd.Series.mean, "std", 'min', 'max'])

Se puede especificar el nombre de las etiquetas de la serie resultante mediante un diccionario.

In [ ]:
df_fifa['Value (M)'].agg({'Media':pd.Series.mean, 'Desviación':"std"}) 

### <font color="#5E5E5E">`DataFrame.agg()`</font>

El método `DataFrame.agg()` recibe un argumento que puede ser una función de agregación, o una lista de funciones de agregación. El parámetro `axis` determina el índice sobre el que actúa, y el resultado depende del número de funciones de agregación utilizadas.

In [ ]:
df_fifa[['Value (M)', 'Wage (K)']].agg('mean', axis=0) # Devuelve una serie

In [ ]:
df_fifa[['Value (M)', 'Wage (K)']].agg(['mean', 'std'], axis=0) # Devuelve un DataFrame

En el caso de los `DataFrame`, el diccionario permite determinar a qué columna se aplica cada método.

In [ ]:
df_fifa[['Value (M)', 'Wage (K)']].agg({'Value (M)':["mean", "std"], 'Wage (K)': "sum"}, axis=0) # Devuelve un DataFrame

Si se utiliza solamente una función por columna, es posible nombrarla las filas resultado.

In [ ]:
df_fifa[['Value (M)', 'Wage (K)']].agg(valor  = ('Value (M)',"mean"), 
                                       sueldo = ('Wage (K)',pd.Series.sum), 
                                       axis=0) # Devuelve un DataFrame

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>  `DataFrame.aggregate()` \ `Series.aggregate()` es el mismo método que `DataFrame.agg()` \ `Series.agg()`, que es un alias.
</div>

### <font color="#5E5E5E">Agregación con `DataFrame.apply()`</font>

El método `DataFrame.apply(función)` recibe como argumento una `función` que opera sobre una `Series`, que puede ser una fila o una columna. Anteriormente se ha visto que dicha función puede operar sobre cada elemento, y devolver una nueva `Series` (y el resultado de `DataFrame.apply()` un `DataFrame`). Pero la función puede, a su vez, *aplicar* una operación definida sobre el conjunto de datos de la `Series`, y  **devolver una agregación**.

In [ ]:
def rango(valores):
    return valores.max()-valores.min()

df_fifa[['Value (M)', 'Wage (K)']].apply(rango, axis=0) # Opera sobre cada columna

<div class="alert alert-block alert-warning">

<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
Algunas funciones no son agregaciones en el sentido numérico de la palabra. Sin embargo, operan con todos los elementos de la `Series` y devuelven un escalar resultado de operar con los valores.
</div>

In [ ]:
def anuncio(fila):
    return f"¡{fila['Name']} cobra {fila['Wage (K)']:.0f} mil € por semana!"

df_fifa.dropna(subset=['Name', 'Wage (K)']).apply(anuncio, axis=1).head()

---
## <font color="#5E5E5E"> 4. Funciones de transformación / agregación en `Series`</font>


Algunas transformaciones calculan cada resultado usando otros elementos de la serie. `cumsum()` calcula acumulados; `diff()` compara un valor con el anterior y `pct_change()` devuelve su cambio relativo. Las tres respetan el orden actual de las filas y generan una nueva `Series`. 

In [ ]:
df_fifa['Value (M)'].head(5).cumsum()
#df_fifa['Value (M)'].head(5).diff()
#df_fifa['Value (M)'].head(5).pct_change()

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>  En la parte del tutorial *pandas* dedicada al manejo de series temporales se verán con más detenimiento estas funciones.
</div>

---
## <font color="#5E5E5E"> <i class="fa fa-plus-circle" aria-hidden="true" style="color:#5E5E5E"></i> Eficiencia</font>

La mejora de la eficiencia debido a la vectorización también se percibe (aunque en menor medida) en las funciones de agregación. En los siguientes ejemplos se suman los elementos de una serie mediante un bucle, primero, y mediante la función vectorizada, después.


In [ ]:
%%timeit -n 10 serie = pd.Series(np.random.randint(0,1000,5000))
suma = 0
for elem in serie:
    suma+=elem

In [ ]:
%%timeit -n 10 serie = pd.Series(np.random.randint(0,1000,5000))
suma = serie.sum()
#suma = np.sum(serie)

Por otra parte, es conveniente utilizar directamente operaciones vectorizadas. En la siguiente prueba, se crean dos columnas y se calcula la media para cada fila. Primero con `DataFrame.apply(axis=1)`, y después directamente.

In [ ]:
df = pd.DataFrame({'A':np.random.randint(0,1000,10000), 'B':np.random.randint(0,1000,10000)})

In [ ]:
def media(fila):
    return (fila['A']+fila['B'])/2

In [ ]:
%%timeit -n 10 
df.apply(media, axis=1)

In [ ]:
%%timeit -n 10 
(df['A']+df['B'])/2

<br>
<div align="right">
<a href="#inicio"><font size=5 color="#B51700"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

---

<div align="right">
<font size=5 color="#B51700"><i class="fa fa-coffee" aria-hidden="true"></i></font>
</div>